# R2 — does more optimisation move the QRY learnability threshold?

**This is a DIAGNOSTIC, not a repair.** Q7/Q8 left two hypotheses standing and
read-only diagnosis cannot separate them:

| | |
|---|---|
| **H8** insufficient optimisation | the model under-fits because ~2 epochs is too few |
| **H7** capacity | the model cannot represent the distribution at all |

Doubling steps discriminates them. Framed as a repair, a failure would throw
that information away.

**The only variable under test is `steps`: 20,000 → 40,000.** Everything else
is held at the `best_e5.pt` baseline — same architecture, same `class_alpha`,
`boundary_weight = 1.0` (S4 stays closed), `max_len` unchanged at 256 (R1 was
demoted on measured cost/benefit, and bundling it here would destroy
attribution).

Registered in `research/qry/PREREG_R2.md`.

> **Cells 1 and 2 are taken from `notebooks/kaggle_train.ipynb`**, which already
> solved two failure modes the hard way: the mount path must be *discovered*,
> not hard-coded, and training must be called **in-process**. An earlier version
> of this notebook hard-coded the path and shelled out, and failed on both.

In [ ]:
# --- locate the dataset ------------------------------------------------------
# Never hard-code the mount path. It is Kaggle's SLUG, derived from the title
# rather than from what you typed, and the depth varies: a dataset can land at
# /kaggle/input/<slug>/ or nested under /kaggle/input/datasets/<user>/<slug>/.
# Search at ANY depth.
#
# Anchor on bundle.manifest.json rather than a data file: it is the one name
# that cannot drift when the corpus layout changes.
from pathlib import Path

root = Path("/kaggle/input")
candidates = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []

if not candidates:
    print("tree under /kaggle/input:")
    for path in (sorted(root.rglob("*"))[:60] if root.exists() else []):
        print("   ", path)
    raise SystemExit(
        "No bundle.manifest.json anywhere under /kaggle/input.\n"
        "  If the tree above is empty the dataset is not attached:\n"
        "  right panel -> Add Input -> search your dataset -> click +\n"
        "  Inputs mount when the SESSION starts, so re-run after attaching.")

DATA = candidates[0].parent
print("dataset:", DATA)

In [ ]:
# --- environment -------------------------------------------------------------
import subprocess, sys, importlib

sys.path.insert(0, str(DATA))

# tldextract carries the Public Suffix List. It may already be in the image;
# only install when genuinely missing, so the run does not depend on the
# network any more than it has to.
try:
    importlib.import_module("tldextract")
    print("tldextract: already present")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tldextract"],
                   check=True)
    print("tldextract: installed")

import torch
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")

In [ ]:
# --- sanity check: the bundle is the two-corpus one, not the old E2 bundle ----
# kaggle_train loads these eight streams BY NAME. Kaggle decompresses gzip
# members on upload, so a stream may arrive as .jsonl rather than .jsonl.gz --
# resolve() accepts either, which is why both are checked.
need = ["name_train", "structure_train", "phishing_name_train",
        "phishing_structure_train", "name_val", "structure_val",
        "phishing_name_val", "phishing_structure_val"]

present = {p.name for p in DATA.iterdir()}
missing = [s for s in need
           if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
for s in need:
    print(("OK   " if s not in missing else "MISS "), s)

pkg = (DATA / "uie" / "kaggle_train.py").is_file()
print("uie package present:", pkg)

if missing or not pkg:
    raise SystemExit(
        f"STOP. missing {missing}, package present={pkg}.\n"
        "  Only train.jsonl/val.jsonl present? That is the OLD E2 bundle,\n"
        "  which predates the two-corpus architecture and is the wrong dataset.")
print("\nready")

In [ ]:
# --- train -------------------------------------------------------------------
# Called IN-PROCESS on purpose. `!python -m uie.kaggle_train` would spawn a
# fresh interpreter that does NOT inherit this notebook's sys.path, so the
# package added two cells above is invisible to the child and the run dies with
# ModuleNotFoundError.
#
# EVERY ARGUMENT IS STATED EXPLICITLY. run()'s own defaults are d_model=256,
# n_layers=6, frame=False -- which would silently train a 4.8M model with no
# frame token. That is NOT comparable to best_e5.pt: the frame token changes the
# vocabulary AND the loss, because it disables field-scoped content masking
# (which is why Q8 H6 measured trained/own = 1.0 for every field).
#
# WATCH THE FIRST THOUSAND STEPS. Loss starts near ln(vocab) ~ 4.6 and should
# fall under 3.0 within a thousand steps. Validation loss is reported PER CLASS
# and is weighted by the same field masks as training, so best.pt is selected on
# the objective actually being optimised. If phishing stalls while benign
# improves, class sampling is not working -- stop rather than burn quota.
#
# ~4h expected on a T4: E5 is ~3x the compute per step of the 4.8M model that
# KAGGLE.md timed at 40 min / 20k steps, and this doubles the steps again.
from pathlib import Path
from uie.kaggle_train import run

result = run(
    DATA,
    Path("/kaggle/working"),
    steps=40000,          # <-- THE ONLY VARIABLE UNDER TEST (baseline 20000)
    batch_size=256,       # baseline
    lr=3e-4,              # baseline
    d_model=384,          # E5 architecture; run() defaults to 256
    n_layers=8,           # E5 architecture; run() defaults to 6
    n_heads=8,
    eval_every=1000,
    class_alpha=1.0,      # exposure-neutral; the memorisation guard stays on
    frame=True,           # E5; run() defaults to False
    factorized=False,     # E6 was REJECTED; S4 builds on E5, not E6
    boundary_weight=1.0,  # S4/S4b stays CLOSED
)
print("\ndone")

In [ ]:
# --- confirm the config actually took effect ---------------------------------
# A run that silently used run()'s defaults would look perfectly healthy in the
# log and be worthless. Read it back from the artifact rather than trusting the
# call site.
import json

cfg = {**result.get("model_config", {}), **result.get("train_config", {})}
expect = {"d_model": 384, "n_layers": 8, "steps": 40000,
          "batch_size": 256, "class_alpha": 1.0, "max_len": 256}

bad = [(k, v, cfg.get(k)) for k, v in expect.items() if cfg.get(k) != v]
for k, v in expect.items():
    got = cfg.get(k)
    print(f"{'OK  ' if got == v else 'WRONG'} {k:<12} expected {v}  got {got}")

print("\nvocab_size:", cfg.get("vocab_size"),
      "- frame tokens are present if this exceeds the no-frame vocabulary")
print("dropped over max_len:", result.get("dropped_over_max_len"))
print("\nMISMATCHES:", bad if bad else "none - config is as registered")

json.dump(result, open("/kaggle/working/r2_result.json", "w"),
          indent=1, default=str)
print("wrote /kaggle/working/r2_result.json")

---

## Download — the one thing that must not go wrong

The *Output* tab will hold `final.pt`, `best.pt`, `train_result.json`,
`r2_result.json`.

> **RENAME ON DOWNLOAD** to **`best_r2_40k.pt`** and **`final_r2_40k.pt`**.
>
> The script writes the same `best.pt` / `final.pt` names every earlier run
> used. **Never let a file named `best.pt` land over anything in `data/`, and
> never overwrite `best_e5.pt`** — it is the frozen baseline that every result
> in `E3_REPORT.md` section 30 is measured against.

## What runs next, locally — nothing is adopted

In the order registered in `PREREG_R2.md`:

1. **`q8_exposure.py`** — parameter-name band profile against the
   faithful-sampler null. **Primary readout.** Does band 5–19 rise from 4.09%
   toward its null of 20.40%?
   - it moves → **H8**, the deficit is exposure-limited
   - it does not → **H7**, capacity becomes the live question and R3 opens
2. **`q7_hazard.py`** on `name_val` **and** `name_train` — hazard at 50–100
   (currently 1.315 val / 1.346 train), and the **train/val gap, which is the
   memorisation kill criterion**. The gap is ~zero today, the signature of
   under-fitting. If more steps makes *train* fit while *val* stays put, that is
   memorisation — the exact failure `class_alpha = 1.0` exists to prevent, and
   the one E2.2 measured at +1.227.
3. **`run_eval.py`** at `n_main 8000, G=5 × D=3` — safety, read against the
   ±0.0141 retraining-noise confound. **benign SUB clears by only 0.011, so a
   SUB movement is NOT attributable from a single run.**

Adoption is a separate decision that neither this notebook nor `PREREG_R2.md`
pre-authorises.